In [1]:
from google.colab import files

# Prompt upload of the model file
uploaded = files.upload()


KeyboardInterrupt: 

In [12]:
import torch
import torch.nn as nn
from torchvision.models import resnet18

# Rebuild model structure (no pretrained weights)
model = resnet18(pretrained=False)

# Replace classifier head (assuming binary: 'glasses' vs 'no-glasses')
model.fc = nn.Linear(model.fc.in_features, 2)

# Load the full model (not just weights)
model = torch.load("resnet18_glasses_full.pt", map_location="cpu", weights_only=False)

# Ensure it's in eval mode and moved to device
model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print("✅ Model loaded and ready.")


/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


✅ Model loaded and ready.


In [4]:

from google.colab import files

# Upload glasses.zip from your local machine
uploaded = files.upload()


Saving glasses.zip to glasses.zip


In [13]:
import zipfile
import os

# Make sure target directory exists
os.makedirs("glasses", exist_ok=True)

# Unzip
with zipfile.ZipFile("glasses.zip", 'r') as zip_ref:
    zip_ref.extractall("glasses")

print("✅ glasses.zip extracted to /glasses")


✅ glasses.zip extracted to /glasses


In [6]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Path to image data
data_dir = "glasses/glasses/image"

# Define transforms (resize and convert to tensor)
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

# Load dataset
dataset = datasets.ImageFolder(root=data_dir, transform=transform)

# Split into train and validation (80-20 split)
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_ds, val_ds = random_split(dataset, [train_size, val_size])

# Example
print(f"Total images: {len(dataset)} | Train: {len(train_ds)} | Validation: {len(val_ds)}")
print(f"Class to index mapping: {dataset.class_to_idx}")


Total images: 2614 | Train: 2091 | Validation: 523
Class to index mapping: {'neg': 0, 'pos': 1}


In [7]:
import torchvision.transforms.functional as TF

# Select 25 samples each from 'neg' (0) and 'pos' (1) in val_ds
neg_indices = [i for i in range(len(val_ds)) if val_ds[i][1] == 0][:25]
pos_indices = [i for i in range(len(val_ds)) if val_ds[i][1] == 1][:25]
selected_indices = neg_indices + pos_indices

# Stack tensors for the selected indices
test_images = torch.stack([val_ds[i][0] for i in selected_indices]).to(device)

# Create Gaussian-blurred baseline
blurred_baseline = torch.stack([
    TF.gaussian_blur(img.cpu(), kernel_size=11) for img in test_images
]).to(device)

print("✅ Test images and blurred baselines are ready. Total:", len(test_images))


✅ Test images and blurred baselines are ready. Total: 50


In [19]:
from torchvision.models import resnet18
from torch.nn import functional as F

# Store features and gradients
feature_maps = []
gradients = []

def forward_hook(module, input, output):
    feature_maps.append(output)

def backward_hook(module, grad_in, grad_out):
    gradients.append(grad_out[0])

# Register hooks on the last convolutional layer
target_layer = model.layer4[1].conv2
target_layer.register_forward_hook(forward_hook)
target_layer.register_backward_hook(backward_hook)

print("✅ Forward and backward hooks registered.")


✅ Forward and backward hooks registered.


In [26]:
import matplotlib.pyplot as plt
import numpy as np
import cv2
import torch

def generate_gradcam(image_tensor, class_idx=None):
    # Clear previous runs
    feature_maps.clear()
    gradients.clear()

    # Forward pass
    output = model(image_tensor.unsqueeze(0))  # Add batch dim
    pred_class = output.argmax(dim=1).item()
    target_class = class_idx if class_idx is not None else pred_class

    # Backward pass
    model.zero_grad()
    output[0, target_class].backward()

    # Get gradients and features
    grads_val = gradients[0].cpu().detach().numpy()[0]
    fmap = feature_maps[0].cpu().detach().numpy()[0]

    # Compute weights and Grad-CAM
    weights = np.mean(grads_val, axis=(1, 2))  # Global average pooling
    cam = np.zeros(fmap.shape[1:], dtype=np.float32)

    for i, w in enumerate(weights):
        cam += w * fmap[i]

    # Normalize
    cam = np.maximum(cam, 0)
    cam -= cam.min()
    cam /= cam.max()
    cam = cv2.resize(cam, (224, 224))

    return cam, pred_class

import os
os.makedirs("gradcam_outputs", exist_ok=True)

for i in range(len(test_images)):
    img_tensor = test_images[i].to(device)
    heatmap, pred_class = generate_gradcam(img_tensor)

    # Plot and save
    plt.figure(figsize=(6, 6))
    plt.imshow(img_tensor.permute(1, 2, 0).cpu().numpy())
    plt.imshow(heatmap, cmap='jet', alpha=0.5)
    plt.title(f"Grad-CAM for class {pred_class}")
    plt.axis('off')
    plt.savefig(f"gradcam_outputs/gradcam_{i:02d}.png", bbox_inches='tight')
    plt.close()  # Don't show inline
    plt.show()
    print(f"Grad-CAM for image {i} saved.")

/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1830: FutureWarning: Using a non-full backward hook when the forward contains multiple autograd Nodes is deprecated and will be removed in future versions. This hook will be missing some grad_input. Please use register_full_backward_hook to get the documented behavior.
  self._maybe_warn_non_full_backward_hook(args, result, grad_fn)


Grad-CAM for image 0 saved.
Grad-CAM for image 1 saved.
Grad-CAM for image 2 saved.


/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1830: FutureWarning: Using a non-full backward hook when the forward contains multiple autograd Nodes is deprecated and will be removed in future versions. This hook will be missing some grad_input. Please use register_full_backward_hook to get the documented behavior.
  self._maybe_warn_non_full_backward_hook(args, result, grad_fn)


Grad-CAM for image 3 saved.
Grad-CAM for image 4 saved.
Grad-CAM for image 5 saved.
Grad-CAM for image 6 saved.
Grad-CAM for image 7 saved.
Grad-CAM for image 8 saved.
Grad-CAM for image 9 saved.
Grad-CAM for image 10 saved.


/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1830: FutureWarning: Using a non-full backward hook when the forward contains multiple autograd Nodes is deprecated and will be removed in future versions. This hook will be missing some grad_input. Please use register_full_backward_hook to get the documented behavior.
  self._maybe_warn_non_full_backward_hook(args, result, grad_fn)


...

Grad-CAM for image 42 saved.
Grad-CAM for image 43 saved.
Grad-CAM for image 44 saved.
Grad-CAM for image 45 saved.


/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1830: FutureWarning: Using a non-full backward hook when the forward contains multiple autograd Nodes is deprecated and will be removed in future versions. This hook will be missing some grad_input. Please use register_full_backward_hook to get the documented behavior.
  self._maybe_warn_non_full_backward_hook(args, result, grad_fn)


Grad-CAM for image 46 saved.
Grad-CAM for image 47 saved.
Grad-CAM for image 48 saved.


/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1830: FutureWarning: Using a non-full backward hook when the forward contains multiple autograd Nodes is deprecated and will be removed in future versions. This hook will be missing some grad_input. Please use register_full_backward_hook to get the documented behavior.
  self._maybe_warn_non_full_backward_hook(args, result, grad_fn)


Grad-CAM for image 49 saved.


In [27]:
import shutil
from google.colab import files

shutil.make_archive("gradcam_outputs", 'zip', "gradcam_outputs")
files.download("gradcam_outputs.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [20]:
!pip install -q captum
from captum.attr import GradientShap


In [21]:
import torchvision.transforms.functional as TF

blurred_baseline = torch.stack([
    TF.gaussian_blur(img.cpu(), kernel_size=11) for img in test_images
]).to(device)


In [22]:
model.eval()
gs = GradientShap(model)

# Compute attributions
attributions, delta = gs.attribute(
    test_images,
    baselines=blurred_baseline,
    target=1,  # class index for "glasses"
    return_convergence_delta=True
)


/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1830: FutureWarning: Using a non-full backward hook when the forward contains multiple autograd Nodes is deprecated and will be removed in future versions. This hook will be missing some grad_input. Please use register_full_backward_hook to get the documented behavior.
  self._maybe_warn_non_full_backward_hook(args, result, grad_fn)


In [16]:
import os
import matplotlib.pyplot as plt
import numpy as np

os.makedirs("shap_outputs", exist_ok=True)

for i in range(len(test_images)):
    img = test_images[i].cpu().numpy().transpose(1, 2, 0)
    attr = attributions[i].cpu().detach().numpy().transpose(1, 2, 0)

    # Convert to grayscale
    attr_gray = np.mean(attr, axis=2)

    # Normalize and clip
    attr_gray = np.clip(attr_gray, -0.5, 0.5)

    # Plot and save
    fig, ax = plt.subplots()
    ax.imshow(img)
    ax.imshow(attr_gray, cmap='seismic', alpha=0.6)
    ax.axis('off')
    ax.set_title(f"Gradient SHAP Explanation {i}")
    plt.savefig(f"shap_outputs/shap_expl_{i}.png", bbox_inches='tight')
    plt.close()


NameError: name 'attributions' is not defined

In [15]:
import matplotlib.pyplot as plt
import time  # for delay between displays

for i in range(len(test_images)):
    img = test_images[i].cpu().numpy().transpose(1, 2, 0)
    attr = attributions[i].cpu().detach().numpy().transpose(1, 2, 0)

    # Convert to grayscale
    attr_gray = np.mean(attr, axis=2)
    attr_gray = np.clip(attr_gray, -0.5, 0.5)

    # Show image and SHAP explanation
    plt.figure(figsize=(6, 6))
    plt.imshow(img)
    plt.imshow(attr_gray, cmap='seismic', alpha=0.6)
    plt.title(f"Gradient SHAP Explanation {i}")
    plt.axis('off')
    plt.show()

    # Optional pause to view each plot (adjust as needed)
    time.sleep(1.5)


NameError: name 'attributions' is not defined

In [33]:
import shutil
from google.colab import files

shutil.make_archive("shap_outputs", 'zip', "shap_outputs")
files.download("shap_outputs.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [43]:
print(f"Image 21 predicted class: {pred_class}")


Image 21 predicted class: 1


In [45]:
# Function to print true vs predicted class
def compare_true_pred_classes(test_images, val_ds_indices):
    for i in range(len(test_images)):
        # Get image path and true label
        image_path, true_label = val_ds[val_ds_indices[i]]
        image_name = os.path.basename(image_path)

        # Get predicted label using SHAP or the model directly
        img_tensor = test_images[i].unsqueeze(0).to(device)
        output = model(img_tensor)
        pred_class = output.argmax(dim=1).item()

        print(f"Image {i}: {image_name}, True: {true_label}, Predicted: {pred_class}")


In [51]:
!pip install lime


In [8]:
from lime import lime_image
from skimage.segmentation import mark_boundaries


In [9]:
def predict_fn(images):
    images = torch.tensor(images.transpose((0, 3, 1, 2)), dtype=torch.float32).to(device)
    with torch.no_grad():
        outputs = model(images)
        return torch.softmax(outputs, dim=1).cpu().numpy()


In [10]:
explainer = lime_image.LimeImageExplainer()


In [29]:
import os
import matplotlib.pyplot as plt
from skimage.segmentation import mark_boundaries

os.makedirs("lime_outputs", exist_ok=True)

for i in range(len(test_images)):
    # Prepare image
    img_tensor = test_images[i].cpu()
    img_np = img_tensor.permute(1, 2, 0).numpy()

    # Get explanation
    explanation = explainer.explain_instance(
        img_np,
        classifier_fn=predict_fn,
        top_labels=1,
        hide_color=0,
        num_samples=500
    )

    # Get explanation mask
    temp, mask = explanation.get_image_and_mask(
        explanation.top_labels[0],
        positive_only=True,
        num_features=5,
        hide_rest=False
    )

       # Get predicted class from LIME explanation
    pred_class = explanation.top_labels[0]
    pred_label = 'glasses' if pred_class == 1 else 'no glasses'


    # Plot and save
    fig, ax = plt.subplots()
    ax.imshow(img_np)
    ax.imshow(mark_boundaries(img_np, mask), alpha=0.6)
    ax.set_title(f"LIME Explanation {i} – Predicted: {pred_label}")
    ax.axis('off')
    plt.tight_layout()
    plt.savefig(f"lime_outputs/lime_expl_{i}.png", bbox_inches='tight')
    plt.close()


  0%|          | 0/500 [00:00<?, ?it/s]

  0%|          | 0/500 [00:00<?, ?it/s]

  0%|          | 0/500 [00:00<?, ?it/s]

  0%|          | 0/500 [00:00<?, ?it/s]

  0%|          | 0/500 [00:00<?, ?it/s]

...

  0%|          | 0/500 [00:00<?, ?it/s]

  0%|          | 0/500 [00:00<?, ?it/s]

  0%|          | 0/500 [00:00<?, ?it/s]

  0%|          | 0/500 [00:00<?, ?it/s]

  0%|          | 0/500 [00:00<?, ?it/s]

In [31]:
import shutil
from google.colab import files

shutil.make_archive("lime_explanations", 'zip', "lime_outputs")
files.download("lime_explanations.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>